# 1. ハードウェアの準備


In [3]:
!nvidia-smi

Tue Jan 27 02:59:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# 2. プロジェクトのデプロイ

In [4]:
from google.colab import drive
import os

# 1. Google Drive をマウント
drive.mount('/content/drive')

# 2. プロジェクトの保存パスを設定
BASE_PATH = '/content/drive/MyDrive/colab-notebooks/card_extraction'
PROJECT_NAME = 'Business-card-information-extraction'
PROJECT_PATH = os.path.join(BASE_PATH, PROJECT_NAME)

# !!! 以下のURLを実際のGitHubリポジトリのアドレスに置き換えてください !!!
REPO_URL = 'https://github.com/miracle-huang/Business-card-information-extraction.git'

if not os.path.exists(PROJECT_PATH):
    %cd $BASE_PATH
    print(f"clone the project to: {PROJECT_PATH}...")
    !git clone $REPO_URL
else:
    print(f"the project is existed: {PROJECT_PATH}，renewing...")
    %cd $PROJECT_PATH
    !git pull

%cd $PROJECT_PATH
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/colab-notebooks/card_extraction
clone the project to: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction...
Cloning into 'Business-card-information-extraction'...
remote: Enumerating objects: 585, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 585 (delta 73), reused 140 (delta 33), pack-reused 401 (from 1)
Receiving objects: 100% (585/585), 243.66 MiB | 13.48 MiB/s, done.
Resolving deltas: 100% (119/119), done.
Updating files: 100% (332/332), done.
/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction
content_recognition  four_angles       requirements.txt
data		     pose_four_points  segmentation_classification


In [ ]:
%cd /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction

/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction


In [5]:
%pwd

'/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction'

# 3. 必要な環境をセットアップ

In [7]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 35.6 MB/s eta 0:00:00


In [ ]:
import os
cpu_count = os.cpu_count()
print(f"number of cpu workers: {cpu_count}")

number of cpu workers: 12


# プロジェクトを起動して実行する

## Segmentation & Classification

In [ ]:
# Step1: Generate card-with-background dataset

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/tools/step1_generate_seg_dataset.py

[DBG] using rigid_seg_synth: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/src/synth/rigid_seg_synth.py
[DBG] seed: 42
[DBG] out_dir: segmentation_classification/assets/seg_step1_test
[INFO] Synth Seg Dataset (Rigid / Rotation-only)
  out_dir  : segmentation_classification/assets/seg_step1_test
  num      : 100
  cards/img: 2~4
  weights  : 2->3.0, 3->3.0, 4->5.0
  fixed_w  : 720
  margin   : 120
  min_gap  : 50
  angle    : 0.0~360.0
  out_size : None x None (None means keep bg size)
  dyn_bg   : True, only_3plus=True, max_scale=4.0
  workers  : 8
  debug    : True
[OK] Done (multiprocessing).
[OK] Step1 done: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/assets/seg_step1_test


In [ ]:
# Step2: split train and validation

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/tools/step2_split_seg_dataset.py

In [ ]:
# Step3: train YOLOv11-seg

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/tools/step3_train_seg_yolo11_dynamic_mix.py

In [ ]:
# Step4.1: build cls dataset

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/tools/step4_generate_upright_cls_dataset.py

[OK] Step4 dataset generated (rotation-only, no crop, no padding)
  cards total: 50, train cards: 40, val cards: 10
  images: train=160, val=40
  out_dir: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/assets/cls_step4_dataset


In [ ]:
# Step4.2: train orientation cls model

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/tools/step4_train_upright_cls_yolo11.py

In [ ]:
# Step5: predict

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/tools/step5_predict_warp_upright_v5.py

[OK] Done. Outputs in: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/segmentation_classification/outputs/step5_v5


# Pose four points

In [ ]:
# Step1: Generate four point dataset

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/step1_gen_kpt_synth.py

[kpt_synth] Starting generation of 20 images with 8 workers...
100% 20/20 [00:04<00:00,  4.24it/s]
[kpt_synth] Done. Results saved to pose_four_points/assets/step1_test


In [ ]:
# Step2: split train and validation

! python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/step2_split_kpt_dataset.py


[split_kpt_dataset] DONE
  SRC pool images: 1000
  Missing labels: 0
  Bad label files skipped: 0
  Valid pairs used: 1000
  Train: 800
  Val:   200
  Output dir: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/assets/step2_train_dataset
  Config saved to: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/assets/step2_train_dataset/dataset_card4kpt.yaml



In [ ]:
# Step3: train four points model

%cd /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points
!python step3_train_kpt_hybrid_colab.py

/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points
--- Colab Environment Setup ---
Working Dir: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points
Script Location: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/step3_train_kpt_hybrid_colab.py
成功加载并修复 YAML 内容，path -> /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/assets/step2_train_dataset
正在加载模型: yolo11x-pose.pt
开始训练...
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/d

In [ ]:
# Step4: predict

%cd /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points
!python step4_predict_and_warp_kpt.py

/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points
[predict] model=/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/run/kpt_hybrid_yolo11x_pose_colab_bs32_epoch_50/weights/best.pt
[predict] input=/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/assets/step1_test/images  images=20
[predict] out=/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/pose_four_points/outputs/step4_result
[1/20] img_000001.jpg: saved 2 crop(s)
[2/20] img_000002.jpg: saved 3 crop(s)
[3/20] img_000003.jpg: saved 4 crop(s)
[4/20] img_000004.jpg: saved 4 crop(s)
[5/20] img_000005.jpg: saved 2 crop(s)
[6/20] img_000006.jpg: saved 4 crop(s)
[7/20] img_000007.jpg: saved 4 crop(s)
[8/20] img_000008.jpg: saved 4 crop(s)
[9/20] img_000009.jpg: saved 4 crop(s)
[10/20] img_000010.jpg: saved 3 crop(s)
[11/20

# Content Recognition

In [9]:
# train business card content recognition

!python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/content_recognition/src/train.py

[Sanity] base path: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/data/business_card_v2
[Sanity] train: images=40, labels_dir=/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/data/business_card_v2/train/labels, missing_labels_in_first_20=0
[Sanity] val: images=5, labels_dir=/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/data/business_card_v2/valid/labels, missing_labels_in_first_5=0
[Sanity] test: images=5, labels_dir=/content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/data/business_card_v2/test/labels, missing_labels_in_first_5=0
[Sanity] nc=5, names=['address', 'company', 'email', 'name', 'phone']
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, 

In [10]:
# predict + ocr

!python /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/content_recognition/src/predict_ocr.py

[Init] Loading YOLO model: /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/content_recognition/runs/detect/train_yolo11x_bs32_img960_epoch50/weights/best.pt
[Init] Loading EasyOCR reader...
Progress: |██████████████████████████████████████████████████| 100.0% CompleteDownloading recognition model, please wait. This may take several minutes depending upon your network connection.
Progress: |██████████████████████████████████████████████████| 100.0% Complete[Process] Found 5 images. Starting inference...
100% 5/5 [00:02<00:00,  1.89it/s]
[Done] Results saved to /content/drive/MyDrive/colab-notebooks/card_extraction/Business-card-information-extraction/content_recognition/runs/detect/ocr_result
